In [1]:
# Step 1: Inspect bronze files - row counts and columns
tables = ["patients", "encounters", "conditions", "organizations", "providers", "payers"]
bronze = {}

for t in tables:
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(f"Files/Bronze/{t}.csv"))
    bronze[t] = df
    print(f"{t}: {df.count():,} rows | {len(df.columns)} columns")
    print(df.columns, "\n")

StatementMeta(, d0d56060-d27a-4a5e-9310-9e7cbdee03c2, 3, Finished, Available, Finished, False)

patients: 15,354 rows | 25 columns
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE'] 

encounters: 1,371,435 rows | 15 columns
['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION'] 

conditions: 866,514 rows | 6 columns
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'] 

organizations: 6,765 rows | 11 columns
['Id', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'LAT', 'LON', 'PHONE', 'REVENUE', 'UTILIZATION'] 

providers: 37,881 rows | 12 columns
['Id', 'ORGANIZATION', 'NAME', 'GENDER', 'SPECIALITY', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'LAT', 'LON', 'UTILIZATION'] 

payers: 10 rows | 21 columns
['Id', 'NAM

In [2]:
# Step 2: Data quality checks on bronze
from pyspark.sql import functions as F

enc = bronze["encounters"]

# 1. Data types Spark detected for key date columns
print("=== Data types ===")
print("encounters:", [(c, t) for c, t in enc.dtypes if c in ("START", "STOP")])
print("patients:", [(c, t) for c, t in bronze["patients"].dtypes if c in ("BIRTHDATE", "DEATHDATE")])

# 2. Duplicate primary keys
print("\n=== Duplicate Ids ===")
for t in ["patients", "encounters", "organizations", "providers", "payers"]:
    df = bronze[t]
    dups = df.count() - df.select("Id").distinct().count()
    print(f"{t}: {dups:,}")

# 3. Null keys in encounters
print("\n=== Null keys in encounters ===")
for c in ["Id", "PATIENT", "ORGANIZATION", "PROVIDER", "PAYER", "START", "STOP"]:
    print(f"{c}: {enc.filter(F.col(c).isNull()).count():,}")

# 4. Orphan foreign keys (child rows with no matching parent)
print("\n=== Orphan keys ===")
def orphans(child, fk, parent):
    parent_ids = bronze[parent].select(F.col("Id").alias("_pk"))
    return (bronze[child]
            .join(parent_ids, bronze[child][fk] == F.col("_pk"), "left_anti")
            .filter(F.col(fk).isNotNull())
            .count())

print("encounters -> patients:", f"{orphans('encounters', 'PATIENT', 'patients'):,}")
print("encounters -> organizations:", f"{orphans('encounters', 'ORGANIZATION', 'organizations'):,}")
print("encounters -> providers:", f"{orphans('encounters', 'PROVIDER', 'providers'):,}")
print("encounters -> payers:", f"{orphans('encounters', 'PAYER', 'payers'):,}")
print("conditions -> encounters:", f"{orphans('conditions', 'ENCOUNTER', 'encounters'):,}")

# 5. Invalid dates and negative costs
print("\n=== Logic checks ===")
print("STOP before START:", f"{enc.filter(F.col('STOP') < F.col('START')).count():,}")
print("Negative claim cost:", f"{enc.filter(F.col('TOTAL_CLAIM_COST') < 0).count():,}")
enc.select(F.min("START").alias("first_encounter"), F.max("START").alias("last_encounter")).show()

# 6. Encounter mix - tells us if we have enough inpatient stays for readmissions
print("=== Encounter class distribution ===")
enc.groupBy("ENCOUNTERCLASS").count().orderBy(F.desc("count")).show()

StatementMeta(, d0d56060-d27a-4a5e-9310-9e7cbdee03c2, 4, Finished, Available, Finished, False)

=== Data types ===
encounters: [('START', 'timestamp'), ('STOP', 'timestamp')]
patients: [('BIRTHDATE', 'date'), ('DEATHDATE', 'date')]

=== Duplicate Ids ===
patients: 0
encounters: 0
organizations: 0
providers: 0
payers: 0

=== Null keys in encounters ===
Id: 0
PATIENT: 0
ORGANIZATION: 0
PROVIDER: 0
PAYER: 0
START: 0
STOP: 0

=== Orphan keys ===
encounters -> patients: 0
encounters -> organizations: 0
encounters -> providers: 0
encounters -> payers: 0
conditions -> encounters: 0

=== Logic checks ===
STOP before START: 0
Negative claim cost: 0
+-------------------+-------------------+
|    first_encounter|     last_encounter|
+-------------------+-------------------+
|1922-04-02 07:59:09|2022-02-05 20:27:36|
+-------------------+-------------------+

=== Encounter class distribution ===
+--------------+------+
|ENCOUNTERCLASS| count|
+--------------+------+
|      wellness|495983|
|    ambulatory|423113|
|    outpatient|234059|
|    urgentcare| 92906|
|     inpatient| 49636|
|     em

In [3]:
# Step 3: Build silver layer - clean, rename, drop PII, save as Delta tables
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Patients: drop direct identifiers (SSN, names, passport, address, exact location)
silver_patients = bronze["patients"].select(
    F.col("Id").alias("patient_id"),
    F.col("BIRTHDATE").alias("birth_date"),
    F.col("DEATHDATE").alias("death_date"),
    F.col("GENDER").alias("gender"),
    F.col("RACE").alias("race"),
    F.col("ETHNICITY").alias("ethnicity"),
    F.col("MARITAL").alias("marital_status"),
    F.col("CITY").alias("city"),
    F.col("COUNTY").alias("county"),
    F.col("STATE").alias("state"),
    F.col("HEALTHCARE_EXPENSES").alias("healthcare_expenses"),
    F.col("HEALTHCARE_COVERAGE").alias("healthcare_coverage")
)

# Encounters: rename + add length-of-stay columns
silver_encounters = bronze["encounters"].select(
    F.col("Id").alias("encounter_id"),
    F.col("START").alias("start_ts"),
    F.col("STOP").alias("stop_ts"),
    F.col("PATIENT").alias("patient_id"),
    F.col("ORGANIZATION").alias("organization_id"),
    F.col("PROVIDER").alias("provider_id"),
    F.col("PAYER").alias("payer_id"),
    F.lower(F.trim(F.col("ENCOUNTERCLASS"))).alias("encounter_class"),
    F.col("CODE").alias("encounter_code"),
    F.col("DESCRIPTION").alias("encounter_description"),
    F.col("BASE_ENCOUNTER_COST").alias("base_encounter_cost"),
    F.col("TOTAL_CLAIM_COST").alias("total_claim_cost"),
    F.col("PAYER_COVERAGE").alias("payer_coverage"),
    F.col("REASONCODE").alias("reason_code"),
    F.col("REASONDESCRIPTION").alias("reason_description")
).withColumn(
    "duration_hours",
    F.round((F.unix_timestamp("stop_ts") - F.unix_timestamp("start_ts")) / 3600, 2)
).withColumn(
    "los_days", F.datediff(F.to_date("stop_ts"), F.to_date("start_ts"))
)

# Conditions (diagnoses)
silver_conditions = bronze["conditions"].select(
    F.col("START").alias("onset_date"),
    F.col("STOP").alias("resolved_date"),
    F.col("PATIENT").alias("patient_id"),
    F.col("ENCOUNTER").alias("encounter_id"),
    F.col("CODE").alias("condition_code"),
    F.col("DESCRIPTION").alias("condition_description")
)

# Organizations (hospitals / clinics) - drop address, phone, exact coordinates
silver_organizations = bronze["organizations"].select(
    F.col("Id").alias("organization_id"),
    F.col("NAME").alias("organization_name"),
    F.col("CITY").alias("city"),
    F.col("STATE").alias("state"),
    F.col("ZIP").alias("zip"),
    F.col("REVENUE").alias("revenue"),
    F.col("UTILIZATION").alias("utilization")
)

# Providers - keep role attributes, drop personal name and address
silver_providers = bronze["providers"].select(
    F.col("Id").alias("provider_id"),
    F.col("ORGANIZATION").alias("organization_id"),
    F.col("GENDER").alias("gender"),
    F.col("SPECIALITY").alias("specialty"),
    F.col("CITY").alias("city"),
    F.col("STATE").alias("state"),
    F.col("UTILIZATION").alias("utilization")
)

# Payers (insurers)
silver_payers = bronze["payers"].select(
    F.col("Id").alias("payer_id"),
    F.col("NAME").alias("payer_name"),
    F.col("AMOUNT_COVERED").alias("amount_covered"),
    F.col("AMOUNT_UNCOVERED").alias("amount_uncovered"),
    F.col("REVENUE").alias("revenue"),
    F.col("UNIQUE_CUSTOMERS").alias("unique_customers")
)

# Save all as Delta tables in the silver schema
silver = {
    "patients": silver_patients,
    "encounters": silver_encounters,
    "conditions": silver_conditions,
    "organizations": silver_organizations,
    "providers": silver_providers,
    "payers": silver_payers,
}

for name, df in silver.items():
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(f"silver.{name}")
    print(f"silver.{name}: {spark.table(f'silver.{name}').count():,} rows saved")

StatementMeta(, d0d56060-d27a-4a5e-9310-9e7cbdee03c2, 5, Finished, Available, Finished, False)

silver.patients: 15,354 rows saved
silver.encounters: 1,371,435 rows saved
silver.conditions: 866,514 rows saved
silver.organizations: 6,765 rows saved
silver.providers: 37,881 rows saved
silver.payers: 10 rows saved
